# Astrodynamics - Module d'atterrissage Eagle 1
## Expérimentations avec un modèle PPO

In [1]:
import os
import mlflow
from stable_baselines3 import PPO
from stable_baselines3.common.evaluation import evaluate_policy
from gymnasium.wrappers import RecordVideo
import gymnasium as gym

env_name = 'LunarLander-v3'
video_folder = "./videos_ppo"
log_dir='./logs_ppo'
SEED=42
experiment_name="lunarlander-ppo"

train_env = gym.make(env_name)
train_env.reset(seed=SEED)
train_env.action_space.seed(SEED)

video_env = gym.make(env_name, render_mode="rgb_array")
video_env.reset(seed=SEED)
video_env.action_space.seed(SEED)
video_env = RecordVideo(
    video_env,
    video_folder=video_folder,
    episode_trigger=lambda ep_id: True,  # enregistre tous les épisodes
    name_prefix="dqn_lunarlander"
)

# --- MLflow ---
mlflow.set_experiment(experiment_name)
model_name = "ppo_lunarlander"
NB_STEPS = 500000

experiment = mlflow.get_experiment_by_name(experiment_name)
print(f"Experiment ID: {experiment.experiment_id}")
print(f"Tracking URI: {mlflow.get_tracking_uri()}")

# --- Hyperparams PPO (simples) ---
hparams = dict(
    policy="MlpPolicy",
    learning_rate=0.003,
    n_steps=2048,
    batch_size=64,
    n_epochs=10,
    gamma=0.99,
    gae_lambda=0.95,
    clip_range=0.2,
    ent_coef=0.0,
    vf_coef=0.5,
    max_grad_norm=0.5,
    tensorboard_log=log_dir,
)

with mlflow.start_run(run_name=f"{model_name}_{NB_STEPS}_steps"):
    mlflow.log_param("nb_steps", NB_STEPS)
    mlflow.log_params(hparams)

    model = PPO(
        policy=hparams["policy"],
        env=train_env,
        verbose=0,
        seed=SEED,
        learning_rate=hparams["learning_rate"],
        n_steps=hparams["n_steps"],
        batch_size=hparams["batch_size"],
        n_epochs=hparams["n_epochs"],
        gamma=hparams["gamma"],
        gae_lambda=hparams["gae_lambda"],
        clip_range=hparams["clip_range"],
        ent_coef=hparams["ent_coef"],
        vf_coef=hparams["vf_coef"],
        max_grad_norm=hparams["max_grad_norm"],
        tensorboard_log=hparams["tensorboard_log"],
    )

    model.learn(total_timesteps=NB_STEPS)

    model_path = f"{model_name}.zip"
    model.save(model_name)  # SB3 ajoute .zip
    if os.path.exists(model_path):
        mlflow.log_artifact(model_path, artifact_path="model")

    mean_reward, std_reward = evaluate_policy(
        model,
        video_env,
        n_eval_episodes=100,
        warn=False
    )

    video_env.close()
    print(f"Récompense moyenne sur 100 épisodes : {mean_reward:.2f} +/- {std_reward:.2f}")
    mlflow.log_metric("eval_mean_reward", float(mean_reward))
    mlflow.log_metric("eval_std_reward", float(std_reward))
    mlflow.end_run()

/usr/local/lib/python3.12/site-packages/pygame/pkgdata.py:25: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_stream, resource_exists
/usr/local/lib/python3.12/site-packages/gymnasium/wrappers/rendering.py:293: UserWarning: WARN: Overwriting existing videos at /workspace/src/mission/videos_ppo folder (try specifying a different `video_folder` for the `RecordVideo` wrapper if this is not desired)
  logger.warn(
2026/01/28 10:06:13 WARNING mlflow.utils.git_utils: Failed to import Git (the Git executable is probably not on your PATH), so Git SHA is not available. Error: Failed to initialize: Bad git executable.
The git executable must be specified in one of the following ways:
    - be included in your $PATH
    - be set via $GIT_PYTHON_GIT_EXECUTABLE
    - expl

Experiment ID: 582404638991706781
Tracking URI: http://mlflow:5000
Récompense moyenne sur 50 épisodes : 221.47 +/- 82.42
🏃 View run ppo_lunarlander_500000_steps at: http://mlflow:5000/#/experiments/582404638991706781/runs/094e0f4097ed48eab184110b1726123b
🧪 View experiment at: http://mlflow:5000/#/experiments/582404638991706781
